In [ ]:
import pandas as pd
import sqlite3
import matplotlib.pyplot as plt 
import seaborn as sns

In [ ]:
from pathlib import Path
db_path = Path("../inventory.db") if Path("../inventory.db").exists() else Path("inventory.db")
conn=sqlite3.connect(db_path)
tables=pd.read_sql_query("select name from sqlite_master where type='table'",conn)

In [ ]:
tables

In [ ]:
for table in tables['name']:
    print("Table Name",table)
    df=pd.read_sql_query(f"select * from {table} limit 5 ",conn)
    display(df)

In [ ]:
d=pd.read_sql_query("select * from vendor_invoice ",conn)
d

In [ ]:
vendor=d[['Quantity','Dollars','Freight']].corr()
sns.heatmap(vendor,annot=True)
plt.show()

plt.scatter(d['Quantity'], d['Freight'], label='Quantity vs Freight')
plt.scatter(d['Dollars'], d['Freight'], label='Dollars vs Freight')

plt.xlabel('Quantity / Dollars')
plt.ylabel('Freight')
plt.legend()
plt.show()

In [ ]:
d['fright_per_unit']=d['Freight']/d['Quantity']

In [ ]:
d

In [ ]:
d[['Dollars','Quantity','Freight']].describe()

In [ ]:
from sklearn.model_selection import train_test_split

X=d[['Dollars','Quantity']]
y=d['Freight']

X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.2,random_state=42)

In [ ]:
X_train

In [ ]:
X_test

In [ ]:
from sklearn.preprocessing import StandardScaler

scaler=StandardScaler()
x_trained=scaler.fit_transform(X_train)
x_tested=scaler.transform(X_test)

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error,mean_squared_error,r2_score

In [ ]:
def detect(model,x_test,y_test,model_name):
    pred=model.predict(x_test)
    mae=mean_absolute_error(y_test,pred)
    mse=mean_squared_error(y_test,pred)
    r2=r2_score(y_test,pred)
    print(f"{model_name}")
    print("MAE:",mae)
    print("MSE:",mse)
    print("RMSE:",mse**0.5)
    print("R2:",r2)

In [ ]:
model1=LinearRegression()
model1.fit(x_trained,y_train)

model2=DecisionTreeRegressor(random_state=42)
model2.fit(x_trained,y_train)

model3=RandomForestRegressor(random_state=42)
model3.fit(x_trained,y_train)

In [ ]:
detect(model1,x_tested,y_test,"Linear Regression")
detect(model2,x_tested,y_test,"Decision Tree")
detect(model3,x_tested,y_test,"Random Forest")